# Phase 3 — SpecAugment for genre classification

This notebook compares the Phase 2 CNN with a matched CNN trained using **SpecAugment**. Both models use the same filtered train/validation/test manifests, cached normalized log-mel spectrograms, six fixed 5-second windows, model architecture, optimizer, epoch limit, early-stopping rule, and track-level probability averaging. The only training-input change is dynamic frequency and time masking for the augmented run.

Augmentation is applied to training batches in memory. It never writes to the feature cache. Validation, test, and inference inputs stay unaugmented. The test set is held back until both model configurations and their checkpoints have been selected using validation results.

The notebook reports track-level accuracy, balanced accuracy, macro-F1, per-genre precision/recall/F1, confusion matrices, and the recall changes for genres that were weakest on baseline validation. After execution, it also writes a JSON experiment record under `artifacts/`.


## Before running

Run this notebook from the project root, or in Colab after the setup cell downloads GTZAN. The project-local filtered manifests are used when available; otherwise the notebook downloads those same repository files. Do not edit the manifests for this experiment.

The Phase 2 log-mel cache must already exist at its original path and with its original settings. This notebook only verifies and reads cache entries; it never creates or changes spectrogram files. If a fresh runtime has no cache, run the Phase 2 feature-cache preparation once first, then return here.

The no-augmentation model is retrained in this notebook with the Phase 2 training setup so the baseline and augmented model can be compared under the same run, seed, split, and best-checkpoint rule. Training both configurations can take several minutes, especially without a GPU.


In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path, PurePosixPath

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
from huggingface_hub import hf_hub_download

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "genres_original"
GENRE_FOLDERS = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]

def genre_file_counts(root):
    return {
        genre: len(list((root / genre).glob("*.wav")))
        for genre in GENRE_FOLDERS
    }

DATA_DIR.mkdir(parents=True, exist_ok=True)
existing_counts = genre_file_counts(DATA_DIR)
if all(count >= 90 for count in existing_counts.values()):
    print(f"GTZAN is already present at {DATA_DIR.resolve()} — skipping download.")
else:
    archive_path = hf_hub_download(
        repo_id="marsyas/gtzan",
        repo_type="dataset",
        filename="data/genres.tar.gz",
        local_dir=str(PROJECT_DIR / "data" / "downloads"),
    )
    print(f"Downloaded archive: {archive_path}")

    extracted_files = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in archive.getmembers():
            if not member.isfile():
                continue
            member_path = PurePosixPath(member.name)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in GTZAN archive: {member.name}")
            filename = member_path.name
            if not filename.lower().endswith(".wav") or filename.startswith("._"):
                continue
            genres_in_path = [
                part.lower() for part in member_path.parts[:-1]
                if part.lower() in GENRE_FOLDERS
            ]
            if not genres_in_path:
                continue
            genre = genres_in_path[-1]
            if genre == "jazz" and filename == "jazz.00054.wav":
                continue
            destination = DATA_DIR / genre / filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            extracted_files += 1

    final_counts = genre_file_counts(DATA_DIR)
    missing_or_incomplete = {genre: count for genre, count in final_counts.items() if count < 90}
    if missing_or_incomplete:
        raise RuntimeError(
            f"GTZAN extraction looks incomplete: {missing_or_incomplete}. "
            "Check the Colab internet connection and rerun this cell."
        )
    print(f"Extracted {extracted_files} usable WAV files into {DATA_DIR.resolve()}")
    print("Files per genre:", final_counts)

print(f"Dataset directory ready: {DATA_DIR.resolve()}")

## 1. Imports, fixed settings, and reproducibility

The augmentation starts conservatively: at most two masks per axis, each no wider than 12 Mel bands or 24 time frames. Since 24 frames are about 0.56 seconds at the configured hop length, each individual mask removes a short region. Mask widths are sampled anew for every training batch and every window.

In [ ]:
import hashlib
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
# Deterministic kernels make the paired comparison easier to reproduce.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Audio and feature settings: unchanged from Phase 2.
PROJECT_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "genres_original"
SAMPLE_RATE = 22_050
TRACK_SECONDS = 30
SEGMENT_SECONDS = 5
N_MELS = 128
N_FFT = 2_048
HOP_LENGTH = 512
DB_RANGE = 80.0

# Training settings: unchanged from Phase 2.
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 6
LEARNING_RATE = 1e-3

# SpecAugment settings. Widths are inclusive upper bounds; each mask can be zero-width.
NUM_FREQ_MASKS = 2
FREQ_MASK_MAX_WIDTH = 12
NUM_TIME_MASKS = 2
TIME_MASK_MAX_WIDTH = 24

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_CHECKPOINT_PATH = ARTIFACTS_DIR / "gtzan_phase3_baseline.pt"
SPECAUGMENT_CHECKPOINT_PATH = ARTIFACTS_DIR / "gtzan_phase3_specaugment.pt"
RESULTS_PATH = ARTIFACTS_DIR / "phase3_specaugment_results.json"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Dataset folder: {DATA_DIR.resolve()}")
print(f"SpecAugment: {NUM_FREQ_MASKS} frequency masks (max width {FREQ_MASK_MAX_WIDTH}), "
      f"{NUM_TIME_MASKS} time masks (max width {TIME_MASK_MAX_WIDTH})")


## 2. Find and index the audio files

This is the Phase 2 indexing step. The manifests below remain authoritative; files omitted from them are excluded from modeling.

In [ ]:
# A few common extraction layouts are checked before asking you to edit DATA_DIR.
if not DATA_DIR.exists():
    alternatives = [
        Path("genres_original"),
        Path("Data/genres_original"),
        Path("data/Data/genres_original"),
    ]
    DATA_DIR = next((path for path in alternatives if path.exists()), DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the GTZAN genre folders at {DATA_DIR.resolve()}. "
        "Extract the dataset and set DATA_DIR to the folder that directly contains "
        "the ten genre folders."
    )

audio_extensions = {".wav", ".au", ".mp3", ".flac", ".ogg"}
genre_dirs = sorted(path for path in DATA_DIR.iterdir() if path.is_dir())
CLASS_NAMES = [path.name for path in genre_dirs]
if len(CLASS_NAMES) != 10:
    raise ValueError(
        f"Expected 10 genre folders, found {len(CLASS_NAMES)}: {CLASS_NAMES}. "
        "Check that DATA_DIR points to the folder containing the genre folders."
    )

class_to_id = {name: index for index, name in enumerate(CLASS_NAMES)}

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as audio_file:
        for block in iter(lambda: audio_file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

records = []
for genre_dir in genre_dirs:
    for audio_path in sorted(genre_dir.iterdir()):
        if audio_path.is_file() and audio_path.suffix.lower() in audio_extensions:
            try:
                audio_info = sf.info(str(audio_path))
                file_hash = sha256_file(audio_path)
            except (OSError, RuntimeError, ValueError) as error:
                print(f"Skipping unreadable audio file {audio_path}: {error}")
                continue
            records.append(
                {
                    "path": audio_path.resolve(),
                    "relative_path": audio_path.relative_to(DATA_DIR).as_posix(),
                    "track_name": audio_path.name,
                    "genre": genre_dir.name,
                    "label_id": class_to_id[genre_dir.name],
                    "duration_seconds": audio_info.duration,
                    "sha256": file_hash,
                }
            )

tracks = pd.DataFrame(records)
if tracks.empty:
    raise ValueError(
        f"No supported audio files found directly inside the genre folders in {DATA_DIR}."
    )

print(f"Found {len(tracks)} audio files across {tracks['genre'].nunique()} genres.")
display(tracks.head())


## 3. Apply the existing filtered manifests

The manifest paths and split membership are unchanged. Exact byte copies are checked so they cannot cross split boundaries.

In [ ]:
from urllib.request import urlopen

SPLIT_MANIFEST_DIR = PROJECT_DIR / "filtered_splits"
MANIFEST_FILES = {
    "train": "train_filtered.txt",
    "validation": "valid_filtered.txt",
    "test": "test_filtered.txt",
}
MANIFEST_BASE_URL = (
    "https://raw.githubusercontent.com/Rani1281/SongGenreClassification/main/filtered_splits"
)


In [ ]:
def read_manifest(filename):
    manifest_path = SPLIT_MANIFEST_DIR / filename
    if not manifest_path.exists():
        download_errors = []
        try:
            with urlopen(f"{MANIFEST_BASE_URL}/{filename}", timeout=30) as response:
                manifest_text = response.read().decode("utf-8-sig")
            SPLIT_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
            manifest_path.write_text(manifest_text, encoding="utf-8")
        except Exception as error:
            download_errors.append(f"{MANIFEST_BASE_URL}: {error}")
            details = "\n".join(download_errors)
            raise RuntimeError(
                f"Could not load {filename} from {SPLIT_MANIFEST_DIR} or GitHub. "
                f"Download the filtered split files into that folder.\n{details}"
            ) from error

    entries = []
    for line_number, line in enumerate(
        manifest_path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        entry = line.strip().replace("\\", "/")
        if not entry or entry.startswith("#") or entry == "jazz/jazz.00054.wav":
            continue
        parts = entry.split("/")
        if len(parts) != 2 or any(part in {"", ".", ".."} for part in parts):
            raise ValueError(
                f"Invalid relative track path in {manifest_path} at line {line_number}: {line!r}"
            )
        entries.append("/".join(parts))

    if not entries:
        raise ValueError(f"The split manifest is empty: {manifest_path}")
    repeated_entries = pd.Index(entries)[pd.Index(entries).duplicated()].unique().tolist()
    if repeated_entries:
        raise ValueError(
            f"Repeated paths in {manifest_path}: {repeated_entries[:10]}"
        )
    return entries

In [ ]:
manifest_entries = {
    split: read_manifest(filename)
    for split, filename in MANIFEST_FILES.items()
}

path_to_split = {}
for split, entries in manifest_entries.items():
    for relative_path in entries:
        if relative_path in path_to_split:
            raise ValueError(
                f"Track {relative_path} appears in both {path_to_split[relative_path]} "
                f"and {split} manifests."
            )
        path_to_split[relative_path] = split

if tracks["relative_path"].duplicated().any():
    duplicate_paths = tracks.loc[
        tracks["relative_path"].duplicated(keep=False), "relative_path"
    ].tolist()
    raise ValueError(f"Dataset contains repeated relative paths: {duplicate_paths[:10]}")

available_paths = set(tracks["relative_path"])
missing_paths = sorted(set(path_to_split) - available_paths)
if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths)} paths in the filtered manifests were not found in DATA_DIR. "
        f"Examples: {missing_paths[:10]}"
    )

indexed_track_count = len(tracks)
tracks["split"] = tracks["relative_path"].map(path_to_split)
tracks = tracks.loc[tracks["split"].notna()].copy().reset_index(drop=True)

selected_conflicts = (
    tracks.groupby("sha256")["genre"].nunique().loc[lambda counts: counts > 1].index
)
if len(selected_conflicts):
    conflicting_selected_tracks = tracks.loc[
        tracks["sha256"].isin(selected_conflicts),
        ["relative_path", "genre", "split", "sha256"],
    ].sort_values(["sha256", "relative_path"])
    display(conflicting_selected_tracks)
    raise ValueError(
        "The filtered manifests include exact copies with conflicting genre labels. "
        "Resolve those paths in the manifests before training."
    )

split_count_per_hash = tracks.groupby("sha256")["split"].nunique()
if split_count_per_hash.max() > 1:
    crossing_hashes = split_count_per_hash[split_count_per_hash > 1].index
    display(
        tracks.loc[tracks["sha256"].isin(crossing_hashes),
                   ["relative_path", "genre", "split", "sha256"]]
        .sort_values(["sha256", "relative_path"])
    )
    raise ValueError("An exact-content group appears in more than one split manifest.")

split_counts = pd.crosstab(tracks["genre"], tracks["split"]).reindex(
    index=CLASS_NAMES,
    columns=["train", "validation", "test"],
    fill_value=0,
)
if (split_counts == 0).any().any():
    raise ValueError(
        "Every genre must appear in all three filtered splits. Split counts:\n"
        f"{split_counts}"
    )

display(split_counts)
print(
    f"Manifest totals — train: {int(split_counts['train'].sum())}, "
    f"validation: {int(split_counts['validation'].sum())}, "
    f"test: {int(split_counts['test'].sum())}"
)
print(
    f"Selected {len(tracks)} of {indexed_track_count} indexed audio files; "
    f"{indexed_track_count - len(tracks)} unlisted files were omitted."
)

train_tracks = tracks.loc[tracks["split"] == "train"].reset_index(drop=True)
validation_tracks = tracks.loc[tracks["split"] == "validation"].reset_index(drop=True)
test_tracks = tracks.loc[tracks["split"] == "test"].reset_index(drop=True)

In [ ]:
# percentages

# Use the predefined GTZAN filtered train/validation/test partitions.
# These partitions account for known dataset issues and help prevent artist overlap between splits, reducing artist-related data leakage.
# The resulting proportions (48% / 21% / 31%) are determined by the predefined partitions, not by a conventional random split.

train_pct = len(train_tracks) / len(tracks) * 100
validation_pct = len(validation_tracks) / len(tracks) * 100
test_pct = len(test_tracks) / len(tracks) * 100

print(f"Train split: {round(train_pct)}%")
print(f"Validation split: {round(validation_pct)}%")
print(f"Test split: {round(test_pct)}%")

## 4. Verify and reuse the Phase 2 cached log-mel spectrograms

The cache path and feature settings match Phase 2: 22,050 Hz mono audio, 30 seconds, 128 Mel bands, 2,048-point FFT, 512-frame hop, 80 dB range, and the same normalization/cache key. This section checks that every selected unique track already has a cached feature with the expected dimensions. It performs no audio decoding, feature extraction, or cache writes.


In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SAMPLES = SAMPLE_RATE * TRACK_SECONDS
EXPECTED_FRAMES = 1 + EXPECTED_SAMPLES // HOP_LENGTH
NUM_SEGMENTS = math.ceil(TRACK_SECONDS / SEGMENT_SECONDS)
FRAMES_PER_SEGMENT = math.ceil(EXPECTED_FRAMES / NUM_SEGMENTS)

FEATURE_CACHE_DIR = (
    ARTIFACTS_DIR
    / "gtzan_log_mel"
    / f"sr{SAMPLE_RATE}_mels{N_MELS}_fft{N_FFT}_hop{HOP_LENGTH}_sec{TRACK_SECONDS}_db{int(DB_RANGE)}"
)

def feature_path_for_hash(file_hash):
    return FEATURE_CACHE_DIR / file_hash[:2] / f"{file_hash}.npy"

unique_tracks = tracks.drop_duplicates("sha256")
missing_features = [
    str(feature_path_for_hash(row.sha256))
    for row in unique_tracks.itertuples(index=False)
    if not feature_path_for_hash(row.sha256).is_file()
]
if missing_features:
    examples = "\n".join(missing_features[:10])
    raise FileNotFoundError(
        f"The Phase 2 feature cache is missing {len(missing_features)} selected entries. "
        "This Phase 3 notebook does not regenerate or modify cached spectrograms. "
        "Run the Phase 2 feature-cache preparation once, then rerun this section.\n"
        f"Missing examples:\n{examples}"
    )

expected_feature_shape = (N_MELS, EXPECTED_FRAMES)
invalid_shapes = []
for row in unique_tracks.itertuples(index=False):
    cached_shape = np.load(
        feature_path_for_hash(row.sha256), mmap_mode="r"
    ).shape
    if cached_shape != expected_feature_shape:
        invalid_shapes.append((row.relative_path, cached_shape))
if invalid_shapes:
    raise ValueError(
        f"Cached features do not match Phase 2 shape {expected_feature_shape}: "
        f"{invalid_shapes[:10]}"
    )

print(f"Verified {len(unique_tracks)} existing cached features.")
print(f"Feature cache (read-only in this notebook): {FEATURE_CACHE_DIR.resolve()}")
print(f"One track feature shape: {expected_feature_shape} (mel bands × time frames)")


## 5. Build unaugmented window datasets and loaders

The dataset always returns the original cached window. SpecAugment is not part of `__getitem__`: it is called only inside the training loop. This keeps validation and test data unchanged and makes a fresh random mask on every training pass.

In [ ]:
class MelWindowDataset(Dataset):
    def __init__(self, track_table):
        self.track_rows = track_table.to_dict(orient="records")
        self.examples = [
            (track_index, segment_index)
            for track_index, row in enumerate(self.track_rows)
            for segment_index in range(NUM_SEGMENTS)
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, item_index):
        track_index, segment_index = self.examples[item_index]
        row = self.track_rows[track_index]
        mel = np.load(feature_path_for_hash(row["sha256"]), mmap_mode="r")

        start = segment_index * FRAMES_PER_SEGMENT
        window = np.array(
            mel[:, start : start + FRAMES_PER_SEGMENT],
            dtype=np.float32,
            copy=True,
        )
        if window.shape[1] < FRAMES_PER_SEGMENT:
            window = np.pad(
                window,
                ((0, 0), (0, FRAMES_PER_SEGMENT - window.shape[1])),
            )

        features = torch.from_numpy(window[None, :, :])
        label = torch.tensor(row["label_id"], dtype=torch.long)
        source_track = torch.tensor(track_index, dtype=torch.long)
        return features, label, source_track

train_dataset = MelWindowDataset(train_tracks)
validation_dataset = MelWindowDataset(validation_tracks)
test_dataset = MelWindowDataset(test_tracks)

# Recreate the loaders for each training run with the same shuffle seed.
def make_loaders(seed):
    train_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=train_generator,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, validation_loader, test_loader

train_loader, validation_loader, test_loader = make_loaders(SEED)
sample_features, sample_label, sample_track = next(iter(train_loader))
print(f"Training windows: {len(train_dataset)}")
print(f"Validation windows: {len(validation_dataset)}")
print(f"Test windows: {len(test_dataset)}")
print(f"Batch tensor shape: {tuple(sample_features.shape)}")
print(f"One window shape: {tuple(sample_features[0].shape)}")


## 6. Define SpecAugment for training batches

Each sample in a batch receives independently sampled frequency and time masks. The masks replace normalized values with zero, the normalized floor used by this feature pipeline. A separate CPU random generator drives mask sampling so adding augmentation does not change the model's initialization, dropout RNG, or the order of shuffled training windows between the paired runs.

In [ ]:
def spec_augment_batch(features, generator, apply_frequency=True, apply_time=True):
    """Mask selected axes in a [batch, channel, mel, frame] tensor."""
    if features.ndim != 4:
        raise ValueError(f"Expected a 4D batch tensor, got shape {tuple(features.shape)}")

    batch_size, _, n_mels, n_frames = features.shape
    freq_mask = torch.zeros((batch_size, n_mels), dtype=torch.bool)
    if apply_frequency:
        freq_max = min(FREQ_MASK_MAX_WIDTH, n_mels)
        freq_widths = torch.randint(
            0, freq_max + 1, (batch_size, NUM_FREQ_MASKS), generator=generator
        )
        freq_starts = (
            torch.rand((batch_size, NUM_FREQ_MASKS), generator=generator)
            * (n_mels - freq_widths + 1)
        ).floor().long()
        mel_positions = torch.arange(n_mels)[None, None, :]
        freq_mask = (
            (mel_positions >= freq_starts[:, :, None])
            & (mel_positions < (freq_starts + freq_widths)[:, :, None])
        ).any(dim=1)

    time_mask = torch.zeros((batch_size, n_frames), dtype=torch.bool)
    if apply_time:
        time_max = min(TIME_MASK_MAX_WIDTH, n_frames)
        time_widths = torch.randint(
            0, time_max + 1, (batch_size, NUM_TIME_MASKS), generator=generator
        )
        time_starts = (
            torch.rand((batch_size, NUM_TIME_MASKS), generator=generator)
            * (n_frames - time_widths + 1)
        ).floor().long()
        frame_positions = torch.arange(n_frames)[None, None, :]
        time_mask = (
            (frame_positions >= time_starts[:, :, None])
            & (frame_positions < (time_starts + time_widths)[:, :, None])
        ).any(dim=1)

    combined_mask = (
        freq_mask[:, None, :, None] | time_mask[:, None, None, :]
    ).to(device=features.device)
    return features.masked_fill(combined_mask, 0.0)

# Shape/immutability spot check: augmentation returns a new tensor and leaves input intact.
probe = sample_features[:2].clone()
probe_before = probe.clone()
probe_augmented = spec_augment_batch(
    probe,
    torch.Generator().manual_seed(SEED + 1),
)
assert probe_augmented.shape == probe.shape
assert torch.equal(probe, probe_before)
print(f"Input shape preserved: {tuple(probe_augmented.shape)}")
print("The source batch was not modified.")


## 7. Visual comparison: original and augmented windows

The examples below are two deterministic 5-second windows from different training genres. Each row shows the same original normalized log-mel window beside frequency-only masking, time-only masking, and both masks together. The plotting calls use a separate fixed random generator, so they do not alter training randomness or cached features.


In [ ]:
# Choose one reproducible training track from each of two distinct genres.
plot_track_rows = (
    train_tracks.groupby("genre", group_keys=False)
    .sample(n=1, random_state=SEED)
    .sample(n=2, random_state=SEED)
)
example_segments = [1, 4]
visualization_generator = torch.Generator().manual_seed(SEED + 202)

fig, axes = plt.subplots(2, 4, figsize=(17, 8), constrained_layout=True)
last_image = None
for example_index, (_, track_row) in enumerate(plot_track_rows.iterrows()):
    track_index = int(track_row.name)
    segment_index = example_segments[example_index]
    item_index = track_index * NUM_SEGMENTS + segment_index
    original_features, _, _ = train_dataset[item_index]
    original_batch = original_features.unsqueeze(0)

    frequency_only = spec_augment_batch(
        original_batch,
        visualization_generator,
        apply_frequency=True,
        apply_time=False,
    )[0, 0].numpy()
    time_only = spec_augment_batch(
        original_batch,
        visualization_generator,
        apply_frequency=False,
        apply_time=True,
    )[0, 0].numpy()
    frequency_and_time = spec_augment_batch(
        original_batch,
        visualization_generator,
        apply_frequency=True,
        apply_time=True,
    )[0, 0].numpy()

    variants = [
        ("Original", original_features[0].numpy()),
        ("Frequency mask", frequency_only),
        ("Time mask", time_only),
        ("Frequency + time", frequency_and_time),
    ]
    for column, (title, image) in enumerate(variants):
        axis = axes[example_index, column]
        last_image = axis.imshow(
            image,
            origin="lower",
            aspect="auto",
            cmap="magma",
            vmin=0.0,
            vmax=1.0,
        )
        if example_index == 0:
            axis.set_title(title)
        if column == 0:
            axis.set_ylabel(
                f"{track_row['genre']} — window {segment_index + 1}\nMel band"
            )
        if example_index == 1:
            axis.set_xlabel("Time frame")

fig.suptitle("SpecAugment examples (same scale for all panels)", fontsize=14)
fig.colorbar(
    last_image,
    ax=axes.ravel().tolist(),
    label="Normalized log-mel value",
    shrink=0.85,
)
plt.show()


## 8. Define the unchanged CNN and track-level prediction helpers

The CNN architecture and six-window probability averaging match Phase 2. Validation and test are passed through the model as-is, with no augmentation.

In [ ]:
class SmallAudioCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.filters = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=0.30),
            nn.Linear(64, number_of_classes),
        )

    def forward(self, inputs):
        return self.classifier(self.filters(inputs))

print(f"Trainable parameters: {sum(p.numel() for p in SmallAudioCNN(len(CLASS_NAMES)).parameters()):,}")

@torch.inference_mode()
def collect_predictions(model, data_loader, dataset):
    model.eval()
    all_probabilities = []
    all_window_labels = []
    all_track_indices = []

    for features, labels, track_indices in data_loader:
        logits = model(features.to(DEVICE))
        all_probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_window_labels.append(labels.numpy())
        all_track_indices.append(track_indices.numpy())

    window_probabilities = np.concatenate(all_probabilities)
    window_true = np.concatenate(all_window_labels)
    window_track_indices = np.concatenate(all_track_indices)

    track_probabilities = np.zeros(
        (len(dataset.track_rows), len(CLASS_NAMES)), dtype=np.float64
    )
    window_counts = np.zeros(len(dataset.track_rows), dtype=np.int64)
    np.add.at(track_probabilities, window_track_indices, window_probabilities)
    np.add.at(window_counts, window_track_indices, 1)
    if np.any(window_counts == 0):
        raise ValueError("At least one track is missing a window prediction.")
    track_probabilities /= window_counts[:, None]

    track_true = np.asarray(
        [row["label_id"] for row in dataset.track_rows], dtype=np.int64
    )
    track_predicted = track_probabilities.argmax(axis=1)
    return {
        "window_probabilities": window_probabilities,
        "window_true": window_true,
        "window_predicted": window_probabilities.argmax(axis=1),
        "track_probabilities": track_probabilities,
        "track_true": track_true,
        "track_predicted": track_predicted,
    }

def track_metrics(result):
    return {
        "accuracy": float(accuracy_score(result["track_true"], result["track_predicted"])),
        "balanced_accuracy": float(
            balanced_accuracy_score(result["track_true"], result["track_predicted"])
        ),
        "macro_f1": float(
            f1_score(
                result["track_true"], result["track_predicted"],
                average="macro", zero_division=0,
            )
        ),
    }

def per_genre_metrics(result):
    report = classification_report(
        result["track_true"],
        result["track_predicted"],
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).T.loc[
        CLASS_NAMES, ["precision", "recall", "f1-score", "support"]
    ].rename(columns={"f1-score": "f1"})


## 9. Train matched baseline and SpecAugment models

Both runs use the Phase 2 checkpoint rule: choose the epoch with the best validation track accuracy, breaking ties by validation macro-F1; reduce the learning rate based on validation accuracy; stop after the same patience. Each run starts from the same seed and shuffled-window order. The test loader is not used during training or checkpoint selection.

In [ ]:
TRAINING_CONFIG = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau(mode='max', factor=0.5, patience=2)",
    "checkpoint_selection": "validation track accuracy, then validation macro-F1",
    "track_aggregation": "mean of six window probability vectors",
}


def train_experiment(run_name, use_specaugment, checkpoint_path):
    seed_everything(SEED)
    train_loader, validation_loader, _ = make_loaders(SEED)
    model = SmallAudioCNN(number_of_classes=len(CLASS_NAMES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    augmentation_generator = torch.Generator().manual_seed(SEED + 1)

    history = []
    best_score = (-1.0, -1.0)
    best_epoch = 0
    best_state = None
    best_validation_metrics = None
    best_validation_per_genre = None
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct_windows = 0
        seen_windows = 0

        for features, labels_batch, _ in train_loader:
            features = features.to(DEVICE)
            labels_batch = labels_batch.to(DEVICE)
            if use_specaugment:
                features = spec_augment_batch(features, augmentation_generator)

            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels_batch)
            loss.backward()
            optimizer.step()

            batch_size = labels_batch.size(0)
            total_loss += loss.item() * batch_size
            correct_windows += (logits.argmax(dim=1) == labels_batch).sum().item()
            seen_windows += batch_size

        validation_result = collect_predictions(
            model, validation_loader, validation_dataset
        )
        validation_metrics = track_metrics(validation_result)
        scheduler.step(validation_metrics["accuracy"])
        epoch_record = {
            "epoch": epoch,
            "train_loss": total_loss / seen_windows,
            "train_window_accuracy": correct_windows / seen_windows,
            "validation_track_accuracy": validation_metrics["accuracy"],
            "validation_balanced_accuracy": validation_metrics["balanced_accuracy"],
            "validation_track_macro_f1": validation_metrics["macro_f1"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_record)

        score = (validation_metrics["accuracy"], validation_metrics["macro_f1"])
        if score > best_score:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            best_validation_metrics = validation_metrics
            best_validation_per_genre = per_genre_metrics(validation_result)
            torch.save(
                {
                    "model_state_dict": best_state,
                    "class_names": CLASS_NAMES,
                    "sample_rate": SAMPLE_RATE,
                    "track_seconds": TRACK_SECONDS,
                    "segment_seconds": SEGMENT_SECONDS,
                    "n_mels": N_MELS,
                    "n_fft": N_FFT,
                    "hop_length": HOP_LENGTH,
                    "use_specaugment": use_specaugment,
                    "specaugment": {
                        "num_freq_masks": NUM_FREQ_MASKS if use_specaugment else 0,
                        "frequency_max_width": FREQ_MASK_MAX_WIDTH if use_specaugment else 0,
                        "num_time_masks": NUM_TIME_MASKS if use_specaugment else 0,
                        "time_max_width_frames": TIME_MASK_MAX_WIDTH if use_specaugment else 0,
                    },
                    "training_config": TRAINING_CONFIG,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1

        print(
            f"{run_name} | epoch {epoch:02d}/{EPOCHS} | "
            f"loss {epoch_record['train_loss']:.4f} | "
            f"train window acc {epoch_record['train_window_accuracy']:.3f} | "
            f"val acc {validation_metrics['accuracy']:.3f} | "
            f"val balanced acc {validation_metrics['balanced_accuracy']:.3f} | "
            f"val macro-F1 {validation_metrics['macro_f1']:.3f}"
        )

        if epochs_without_improvement >= PATIENCE:
            print(f"{run_name}: early stopping after {PATIENCE} epochs without improvement.")
            break

    if best_state is None:
        raise RuntimeError(f"{run_name} did not produce a validation checkpoint.")

    model.load_state_dict(best_state)
    model.eval()
    print(f"{run_name}: best validation checkpoint at epoch {best_epoch} ({checkpoint_path})")
    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "validation_metrics": best_validation_metrics,
        "validation_per_genre": best_validation_per_genre,
        "checkpoint_path": str(checkpoint_path),
        "use_specaugment": use_specaugment,
    }

baseline_run = train_experiment(
    "Phase 2 baseline", False, BASELINE_CHECKPOINT_PATH
)
specaugment_run = train_experiment(
    "SpecAugment", True, SPECAUGMENT_CHECKPOINT_PATH
)


## 10. Compare validation results and inspect weak-recall genres

Validation is the experiment-decision split. The configuration decision prioritizes balanced accuracy (average recall across genres), then macro-F1, then accuracy. The three genres with the lowest recall for the baseline's selected validation checkpoint are called out explicitly. The held-out test set is not used for this decision.

In [ ]:
validation_comparison = pd.DataFrame(
    {
        "baseline": baseline_run["validation_metrics"],
        "specaugment": specaugment_run["validation_metrics"],
    }
).T
validation_comparison["delta_specaugment_minus_baseline"] = (
    validation_comparison["specaugment"] - validation_comparison["baseline"]
)
display(validation_comparison.round(3))

baseline_validation_genres = baseline_run["validation_per_genre"]
specaugment_validation_genres = specaugment_run["validation_per_genre"]
weak_validation_genres = baseline_validation_genres["recall"].nsmallest(3).index.tolist()
weak_recall_validation = pd.DataFrame(
    {
        "baseline_recall": baseline_validation_genres.loc[weak_validation_genres, "recall"],
        "specaugment_recall": specaugment_validation_genres.loc[weak_validation_genres, "recall"],
    }
)
weak_recall_validation["delta"] = (
    weak_recall_validation["specaugment_recall"]
    - weak_recall_validation["baseline_recall"]
)
print("Three lowest-recall genres on baseline validation:")
display(weak_recall_validation.sort_values("baseline_recall").round(3))

validation_order_baseline = (
    validation_comparison.loc["baseline", ["balanced_accuracy", "macro_f1", "accuracy"]]
    .to_numpy()
)
validation_order_specaugment = (
    validation_comparison.loc["specaugment", ["balanced_accuracy", "macro_f1", "accuracy"]]
    .to_numpy()
)
selected_configuration = (
    "specaugment"
    if tuple(validation_order_specaugment) > tuple(validation_order_baseline)
    else "baseline"
)
print(
    "Validation decision (balanced accuracy, then macro-F1, then accuracy): "
    f"{selected_configuration}"
)


## 11. Review training curves

The validation curves show the same track-level metrics used to compare the configurations. A widening gap between training loss and validation quality can suggest overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run, label in [(baseline_run, "Baseline"), (specaugment_run, "SpecAugment")]:
    history = run["history"]
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label=label)
    axes[1].plot(
        history["epoch"], history["validation_balanced_accuracy"],
        marker="o", label=f"{label} balanced accuracy",
    )
    axes[1].plot(
        history["epoch"], history["validation_track_macro_f1"],
        marker="s", linestyle="--", label=f"{label} macro-F1",
    )
axes[0].set_title("Training loss (5-second windows)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].set_title("Validation (averaged 30-second tracks)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


## 12. Final held-out test evaluation

Both selected validation checkpoints are evaluated now, after the validation-based experiment decision. Test results estimate final generalization and must not be used to retune masks, epochs, or the selection rule. All reported metrics are track-level, with the same mean-of-six-window-probabilities aggregation.

In [ ]:
_, _, test_loader = make_loaders(SEED)
baseline_test_result = collect_predictions(
    baseline_run["model"], test_loader, test_dataset
)
specaugment_test_result = collect_predictions(
    specaugment_run["model"], test_loader, test_dataset
)

baseline_test_metrics = track_metrics(baseline_test_result)
specaugment_test_metrics = track_metrics(specaugment_test_result)
test_comparison = pd.DataFrame(
    {
        "baseline": baseline_test_metrics,
        "specaugment": specaugment_test_metrics,
    }
).T
test_comparison["delta_specaugment_minus_baseline"] = (
    test_comparison["specaugment"] - test_comparison["baseline"]
)
print("Held-out TEST metrics — one prediction per full track")
display(test_comparison.round(3))


### Per-genre test performance and confusion matrices

The full per-genre table includes precision, recall, and F1 for both models and the SpecAugment-minus-baseline change. Rows marked as weak-recall genres were identified from baseline **validation** recall before test evaluation.

In [ ]:
baseline_test_genres = per_genre_metrics(baseline_test_result)
specaugment_test_genres = per_genre_metrics(specaugment_test_result)
per_genre_test_comparison = pd.concat(
    {
        "baseline": baseline_test_genres,
        "specaugment": specaugment_test_genres,
    },
    axis=1,
)
for metric in ["precision", "recall", "f1"]:
    per_genre_test_comparison[("delta", metric)] = (
        per_genre_test_comparison[("specaugment", metric)]
        - per_genre_test_comparison[("baseline", metric)]
    )
per_genre_test_comparison[("validation_focus", "weak_recall")] = [
    "yes" if genre in weak_validation_genres else "" for genre in CLASS_NAMES
]
display(per_genre_test_comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for axis, result, title in [
    (axes[0], baseline_test_result, "Baseline"),
    (axes[1], specaugment_test_result, "SpecAugment"),
]:
    ConfusionMatrixDisplay.from_predictions(
        result["track_true"],
        result["track_predicted"],
        labels=np.arange(len(CLASS_NAMES)),
        display_labels=CLASS_NAMES,
        cmap="Blues",
        xticks_rotation=45,
        values_format="d",
        ax=axis,
        colorbar=False,
    )
    axis.set_title(f"{title}: test confusion matrix (track-level)")
plt.tight_layout()
plt.show()


## 13. Record the experiment results

This section saves the augmentation settings, fixed training configuration, best validation metrics and per-genre validation scores, test metrics, and per-genre test deltas against the matched baseline. It also records which configuration validation preferred and the baseline's three weakest validation-recall genres.

In [ ]:
def records_by_genre(frame):
    return frame.rename_axis("genre").reset_index().to_dict(orient="records")

experiment_record = {
    "phase": 3,
    "seed": SEED,
    "manifests": MANIFEST_FILES,
    "split_counts_by_genre": split_counts.to_dict(),
    "feature_config": {
        "sample_rate": SAMPLE_RATE,
        "track_seconds": TRACK_SECONDS,
        "segment_seconds": SEGMENT_SECONDS,
        "number_of_segments": NUM_SEGMENTS,
        "n_mels": N_MELS,
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "db_range": DB_RANGE,
        "cache_directory": str(FEATURE_CACHE_DIR),
    },
    "specaugment_config": {
        "frequency_masks_per_window": NUM_FREQ_MASKS,
        "frequency_max_width_mel_bands": FREQ_MASK_MAX_WIDTH,
        "time_masks_per_window": NUM_TIME_MASKS,
        "time_max_width_frames": TIME_MASK_MAX_WIDTH,
        "time_max_width_seconds_approx": TIME_MASK_MAX_WIDTH * HOP_LENGTH / SAMPLE_RATE,
        "fill_value": 0.0,
        "training_only": True,
    },
    "training_config": TRAINING_CONFIG,
    "best_validation": {
        "baseline": {
            "best_epoch": baseline_run["best_epoch"],
            "metrics": baseline_run["validation_metrics"],
            "per_genre": records_by_genre(baseline_run["validation_per_genre"]),
            "checkpoint": baseline_run["checkpoint_path"],
        },
        "specaugment": {
            "best_epoch": specaugment_run["best_epoch"],
            "metrics": specaugment_run["validation_metrics"],
            "per_genre": records_by_genre(specaugment_run["validation_per_genre"]),
            "checkpoint": specaugment_run["checkpoint_path"],
        },
    },
    "validation_decision": {
        "rule": "higher balanced accuracy, then macro-F1, then accuracy",
        "selected_configuration": selected_configuration,
        "weakest_baseline_validation_recall_genres": weak_validation_genres,
        "weak_genre_validation_recall_comparison": (
            weak_recall_validation.rename_axis("genre").reset_index().to_dict(orient="records")
        ),
    },
    "final_test": {
        "baseline": {
            "metrics": baseline_test_metrics,
            "per_genre": records_by_genre(baseline_test_genres),
        },
        "specaugment": {
            "metrics": specaugment_test_metrics,
            "per_genre": records_by_genre(specaugment_test_genres),
        },
        "delta_specaugment_minus_baseline": {
            "metrics": test_comparison.loc[
                "specaugment", ["accuracy", "balanced_accuracy", "macro_f1"]
            ].sub(test_comparison.loc["baseline", ["accuracy", "balanced_accuracy", "macro_f1"]]).to_dict(),
            "per_genre": records_by_genre(
                per_genre_test_comparison["delta"].rename(columns={"f1": "f1"})
            ),
        },
    },
}


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")

RESULTS_PATH.write_text(
    json.dumps(experiment_record, indent=2, default=json_default),
    encoding="utf-8",
)
print(f"Results recorded at: {RESULTS_PATH.resolve()}")
print(f"Validation-selected configuration: {selected_configuration}")
print("Review the weak-recall validation rows and the full per-genre test deltas above.")
